In [ ]:
# Install dependencies
!pip install -q transformers datasets scipy torch pandas huggingface_hub

import hashlib
import json
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from scipy.stats import kendalltau, spearmanr
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Optional HF Auth
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])

CONFIG = {
    "num_student_layers": 4,     # Keep first 4 layers from target
    "head_hidden": 768,
    "n_tweets": 20000,
    "eval_size": 2000,
    "oasis_path": None,
    "oasis_repeat": 5,
    "batch_size": 64,
    "freeze_epochs": 3,
    "finetune_epochs": 6,
    "freeze_lr": 5e-4,           # Lowered for grey-box stability
    "finetune_lr": 2e-5,
    "mse_weight": 0.5,
    "ranking_weight": 2.0,
    "temperature": 0.05,
    "pooler_path": "target_pooler.pt",
    "cache_path": "target_embeddings.pt",
    "out_dir": "surrogate_model",
}

In [ ]:
# Target oracle
def sync_pooler(model, path: str):
    if os.path.exists(path):
        state = torch.load(path, map_location="cpu")
        model.pooler.load_state_dict(state)
    else:
        torch.save({k: v.detach().cpu() for k, v in model.pooler.state_dict().items()}, path)

class TargetEncoder:
    def __init__(self, model_name: str = "Twitter/twhin-bert-base",
                 pooler_path: str | None = None):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        if pooler_path is not None:
            sync_pooler(self.model, pooler_path)
        self.model = self.model.to(DEVICE).eval()

    def pooler_fingerprint(self) -> str:
        h = hashlib.md5()
        for name, tensor in sorted(self.model.pooler.state_dict().items()):
            h.update(name.encode("utf-8"))
            h.update(tensor.detach().cpu().numpy().tobytes())
        return h.hexdigest()

    @torch.no_grad()
    def encode(self, texts: list[str], batch_size: int = 128) -> torch.Tensor:
        outputs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True,
                                    truncation=True, max_length=128).to(DEVICE)
            outputs.append(self.model(**inputs).pooler_output.cpu())
        return torch.cat(outputs, dim=0)

In [ ]:
# Corpora and Dataset
OASIS_TEXT_KEYS = ("description", "user_char", "bio", "persona", "text", "content")

def read_oasis_texts(path: str) -> list[str]:
    texts = []
    if path.lower().endswith(".json"):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        rows = data if isinstance(data, list) else list(data.values()) if isinstance(data, dict) else []
        for row in rows:
            if isinstance(row, dict):
                for key in OASIS_TEXT_KEYS:
                    value = row.get(key)
                    if isinstance(value, str) and value.strip():
                        texts.append(value.strip())
    else:
        import pandas as pd
        df = pd.read_csv(path)
        used = [k for k in OASIS_TEXT_KEYS if k in df.columns]
        for key in used:
            texts += [str(t).strip() for t in df[key].dropna() if str(t).strip()]
    return texts

def build_corpora(n_tweets: int, eval_size: int, oasis_path: str | None,
                  oasis_repeat: int, oasis_eval_frac: float = 0.2) -> dict:
    ds = load_dataset("cardiffnlp/tweet_eval", "sentiment", split="train")
    tweets = list(dict.fromkeys(t.strip() for t in ds["text"]
                                if isinstance(t, str) and t.strip()))
    rng = random.Random(SEED)
    rng.shuffle(tweets)
    tweets = tweets[:n_tweets]
    eval_generic, train = tweets[:eval_size], tweets[eval_size:]
    eval_oasis = []

    if oasis_path is not None and os.path.exists(oasis_path):
        oasis = list(dict.fromkeys(read_oasis_texts(oasis_path)))
        if oasis:
            rng.shuffle(oasis)
            n_eval = int(len(oasis) * oasis_eval_frac)
            eval_oasis, oasis_train = oasis[:n_eval], oasis[n_eval:]
            train = train + oasis_train * oasis_repeat
            rng.shuffle(train)

    return {"train": train, "eval_generic": eval_generic, "eval_oasis": eval_oasis}

def get_target_vectors(target: TargetEncoder, texts: list[str], cache_path: str) -> torch.Tensor:
    key = hashlib.md5(("\n".join(texts) + target.pooler_fingerprint()).encode("utf-8")).hexdigest()
    if os.path.exists(cache_path):
        cached = torch.load(cache_path)
        if isinstance(cached, dict) and cached.get("key") == key:
            return cached["vecs"]
    vecs = target.encode(texts)
    torch.save({"key": key, "vecs": vecs}, cache_path)
    return vecs

class DistillationDataset(Dataset):
    def __init__(self, texts: list[str], target_embeddings: torch.Tensor,
                 tokenizer, max_length: int = 128):
        assert len(texts) == len(target_embeddings)
        encoded = tokenizer(texts, truncation=True, max_length=max_length)
        self.input_ids = encoded["input_ids"]
        self.attention_mask = encoded["attention_mask"]
        self.target_embeddings = target_embeddings

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "target": self.target_embeddings[idx],
        }

def make_collate_fn(tokenizer):
    def collate(batch):
        features = [{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]}
                    for b in batch]
        padded = tokenizer.pad(features, return_tensors="pt")
        targets = torch.stack([b["target"] for b in batch])
        return padded, targets
    return collate

In [ ]:
# Student (surrogate) - Grey-Box Layer Truncation
class SurrogateEncoder(nn.Module):
    def __init__(self, target_model: AutoModel = None, target_tokenizer: AutoTokenizer = None,
                 num_student_layers: int = 4, target_dim: int = 768, head_hidden: int = 768,
                 load_from: str = None):
        super().__init__()
        self.target_dim = target_dim
        self.head_hidden = head_hidden

        if load_from is not None:
            # Load from saved directory
            backbone_dir = os.path.join(load_from, "backbone")
            self.tokenizer = AutoTokenizer.from_pretrained(backbone_dir)
            self.backbone = AutoModel.from_pretrained(backbone_dir).to(DEVICE)
        else:
            # Grey-box initialization from target
            self.tokenizer = target_tokenizer
            student_config = target_model.config
            student_config.num_hidden_layers = num_student_layers
            self.backbone = AutoModel.from_config(student_config).to(DEVICE)

            # Copy embeddings and first N layers
            self.backbone.embeddings.load_state_dict(target_model.embeddings.state_dict())
            for i in range(num_student_layers):
                self.backbone.encoder.layer[i].load_state_dict(target_model.encoder.layer[i].state_dict())

        student_dim = self.backbone.config.hidden_size
        self.projection = nn.Sequential(
            nn.Linear(student_dim, head_hidden),
            nn.GELU(),
            nn.Linear(head_hidden, target_dim),
        ).to(DEVICE)

        if load_from is not None:
            ckpt = torch.load(os.path.join(load_from, "projection.pt"), map_location=DEVICE)
            self.projection.load_state_dict(ckpt["state"])

    @staticmethod
    def _mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._mean_pool(outputs.last_hidden_state, attention_mask)
        return self.projection(pooled)

    def set_backbone_trainable(self, trainable: bool):
        for p in self.backbone.parameters():
            p.requires_grad = trainable

    def save(self, out_dir: str):
        os.makedirs(out_dir, exist_ok=True)
        backbone_dir = os.path.join(out_dir, "backbone")
        self.backbone.save_pretrained(backbone_dir)
        self.tokenizer.save_pretrained(backbone_dir)
        torch.save({"state": self.projection.state_dict(),
                    "target_dim": self.target_dim,
                    "head_hidden": self.head_hidden},
                   os.path.join(out_dir, "projection.pt"))

    @classmethod
    def load(cls, in_dir: str) -> "SurrogateEncoder":
        return cls(load_from=in_dir).eval()

@torch.no_grad()
def encode_texts(model: SurrogateEncoder, texts: list[str], batch_size: int = 128) -> torch.Tensor:
    was_training = model.training
    model.eval()
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = model.tokenizer(batch, return_tensors="pt", padding=True,
                                 truncation=True, max_length=128).to(DEVICE)
        outputs.append(model(inputs["input_ids"], inputs["attention_mask"]).cpu())
    model.train(was_training)
    return torch.cat(outputs, dim=0)

In [ ]:
# Ranking distillation and Training loop
def ranking_distillation_loss(student_pred: torch.Tensor, target_vecs: torch.Tensor,
                              temperature: float) -> torch.Tensor:
    batch_size = student_pred.size(0)
    if batch_size < 2:
        return torch.zeros((), device=student_pred.device)

    with torch.autocast(device_type=student_pred.device.type, enabled=False):
        s = F.normalize(student_pred.float(), dim=-1)
        t = F.normalize(target_vecs.float(), dim=-1)
        sim_s = s @ s.t()
        sim_t = t @ t.t()

        off_diag = ~torch.eye(batch_size, dtype=torch.bool, device=sim_s.device)
        sim_s = sim_s[off_diag].view(batch_size, batch_size - 1)
        sim_t = sim_t[off_diag].view(batch_size, batch_size - 1)

        log_p_student = F.log_softmax(sim_s / temperature, dim=1)
        p_target = F.softmax(sim_t / temperature, dim=1)
        return F.kl_div(log_p_student, p_target, reduction="batchmean")

def _run_phase(student, train_loader, epochs, lr, train_backbone, warmup_ratio,
               use_amp, amp_dtype, scaler, mse_weight, ranking_weight, temperature):
    student.set_backbone_trainable(train_backbone)
    params = [p for p in student.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr)

    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(warmup_ratio * total_steps),
        num_training_steps=total_steps)

    student.train()
    phase_name = "full-finetune" if train_backbone else "frozen-backbone"
    for epoch in range(epochs):
        total_mse, total_rank, skipped = 0.0, 0.0, 0
        for batch, target_vecs in train_loader:
            batch = batch.to(DEVICE)
            target_vecs = target_vecs.to(DEVICE)

            optimizer.zero_grad()
            with torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=use_amp):
                pred = student(batch["input_ids"], batch["attention_mask"])
                mse_loss = F.mse_loss(pred.float(), target_vecs.float())
                rank_loss = ranking_distillation_loss(pred, target_vecs, temperature)
                loss = mse_weight * mse_loss + ranking_weight * rank_loss

            if not torch.isfinite(loss):
                skipped += 1
                continue

            if use_amp and amp_dtype == torch.float16:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            scheduler.step()

            n = target_vecs.size(0)
            total_mse += mse_loss.item() * n
            total_rank += rank_loss.item() * n

        n_total = len(train_loader.dataset)
        msg = (f"[{phase_name}] epoch {epoch + 1}/{epochs} - "
               f"mse: {total_mse / n_total:.6f}  rank_kl: {total_rank / n_total:.6f}")
        if skipped:
            msg += f"  (skipped non-finite batches: {skipped})"
        print(msg)

def train_surrogate(student, train_loader, freeze_epochs, finetune_epochs,
                    freeze_lr, finetune_lr, mse_weight, ranking_weight, temperature,
                    warmup_ratio: float = 0.1):
    use_amp = DEVICE.type == "cuda"
    amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

    print("Phase 1: training projection head only (backbone frozen)")
    _run_phase(student, train_loader, freeze_epochs, freeze_lr, False, warmup_ratio,
               use_amp, amp_dtype, scaler, mse_weight, ranking_weight, temperature)

    print("Phase 2: fine-tuning full model (backbone unfrozen)")
    _run_phase(student, train_loader, finetune_epochs, finetune_lr, True, warmup_ratio,
               use_amp, amp_dtype, scaler, mse_weight, ranking_weight, temperature)

@torch.no_grad()
def evaluate_fidelity(student, target, eval_texts, name="eval", k=10,
                      n_queries=50, candidates_per_query=200):
    if len(eval_texts) < k + 2:
        return None

    n = len(eval_texts)
    candidates = min(candidates_per_query, n - 1)
    target_vecs = target.encode(eval_texts)
    student_vecs = encode_texts(student, eval_texts)

    rng = random.Random(SEED)
    cos_sim = F.cosine_similarity(target_vecs, student_vecs).mean().item()
    shift = rng.randint(1, n - 1)
    baseline = F.cosine_similarity(target_vecs, torch.roll(target_vecs, shifts=shift, dims=0)).mean().item()

    taus, rhos, overlaps, maes = [], [], [], []
    for _ in range(n_queries):
        pool = rng.sample(range(n), candidates + 1)
        user_idx, post_idx = pool[0], pool[1:]
        scores_t = F.cosine_similarity(target_vecs[user_idx].unsqueeze(0), target_vecs[post_idx])
        scores_s = F.cosine_similarity(student_vecs[user_idx].unsqueeze(0), student_vecs[post_idx])

        tau, _ = kendalltau(scores_t.numpy(), scores_s.numpy())
        rho, _ = spearmanr(scores_t.numpy(), scores_s.numpy())
        topk_t = set(torch.topk(scores_t, k).indices.tolist())
        topk_s = set(torch.topk(scores_s, k).indices.tolist())

        taus.append(tau)
        rhos.append(rho)
        overlaps.append(len(topk_t & topk_s) / k)
        maes.append((scores_t - scores_s).abs().mean().item())

    return {
        "n_eval_texts": n,
        "mean_cosine_fidelity": cos_sim,
        "baseline_cosine_fidelity": baseline,
        "kendall_tau_mean": float(np.mean(taus)),
        "kendall_tau_std": float(np.std(taus)),
        "spearman_rho_mean": float(np.mean(rhos)),
        "spearman_rho_std": float(np.std(rhos)),
        f"top_{k}_overlap_mean": float(np.mean(overlaps)),
        f"top_{k}_overlap_std": float(np.std(overlaps)),
        "score_abs_error_mean": float(np.mean(maes)),
    }

In [ ]:
# Main execution (Grey-box)
if __name__ == "__main__":
    cfg = CONFIG
    torch.manual_seed(SEED)

    target = TargetEncoder(pooler_path=cfg["pooler_path"])

    corpora = build_corpora(cfg["n_tweets"], cfg["eval_size"],
                            cfg["oasis_path"], cfg["oasis_repeat"])
    train_texts = corpora["train"]
    train_vecs = get_target_vectors(target, train_texts, cfg["cache_path"])

    # Initialize grey-box student with target model layers
    student = SurrogateEncoder(
        target_model=target.model,
        target_tokenizer=target.tokenizer,
        num_student_layers=cfg["num_student_layers"],
        head_hidden=cfg["head_hidden"]
    )

    train_ds = DistillationDataset(train_texts, train_vecs, student.tokenizer)
    train_loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"], shuffle=True,
        collate_fn=make_collate_fn(student.tokenizer),
        num_workers=0, pin_memory=(DEVICE.type == "cuda"))

    train_surrogate(student, train_loader,
                    cfg["freeze_epochs"], cfg["finetune_epochs"],
                    cfg["freeze_lr"], cfg["finetune_lr"],
                    cfg["mse_weight"], cfg["ranking_weight"], cfg["temperature"])

    for split_name in ("eval_generic", "eval_oasis"):
        if corpora[split_name]:
            result = evaluate_fidelity(student, target, corpora[split_name], name=split_name)
            if result is not None:
                print(f"\n[{split_name}]")
                for key, value in result.items():
                    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

    student.save(cfg["out_dir"])
    print(f"\nSaved grey-box surrogate to '{cfg['out_dir']}'.")

config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Phase 1: training projection head only (backbone frozen)
[frozen-backbone] epoch 1/3 - mse: 0.076686  rank_kl: 0.482565
[frozen-backbone] epoch 2/3 - mse: 0.051755  rank_kl: 0.385849
[frozen-backbone] epoch 3/3 - mse: 0.050605  rank_kl: 0.355020
Phase 2: fine-tuning full model (backbone unfrozen)
[full-finetune] epoch 1/6 - mse: 0.048656  rank_kl: 0.319347
[full-finetune] epoch 2/6 - mse: 0.045501  rank_kl: 0.255622
[full-finetune] epoch 3/6 - mse: 0.044736  rank_kl: 0.218532
[full-finetune] epoch 4/6 - mse: 0.044308  rank_kl: 0.192512
[full-finetune] epoch 5/6 - mse: 0.044116  rank_kl: 0.175261
[full-finetune] epoch 6/6 - mse: 0.044195  rank_kl: 0.163960

[eval_generic]
  n_eval_texts: 2000
  mean_cosine_fidelity: 0.7849
  baseline_cosine_fidelity: 0.7156
  kendall_tau_mean: 0.7152
  kendall_tau_std: 0.1144
  spearman_rho_mean: 0.8801
  spearman_rho_std: 0.1147
  top_10_overlap_mean: 0.4500
  top_10_overlap_std: 0.1723
  score_abs_error_mean: 0.0960


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved grey-box surrogate to 'surrogate_model'.
